In [ ]:
# ==================================================================
# USER CONFIGURATION - Edit these values for your use case
# ==================================================================
VARIABLES = ["2m_temperature"]
YEARS = ["2023"]
MONTHS = ["07"]
DAYS = ["01"]
HOURS = ["12:00"]
BBOX = [55, -8, 49, 2]  # [north, west, south, east] - UK
OUTPUT_DIR = "../data/era5-single-levels"
OUTPUT_FILENAME = "era5_single_levels_quickstart.nc"
# ==================================================================

In [ ]:
# Imports with version checks so users can see at a glance which stack
# the notebook ran against.
import sys
from pathlib import Path

import cdsapi
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

print(f"Python       {sys.version.split()[0]}")
print(f"cdsapi       {cdsapi.__version__}")
print(f"xarray       {xr.__version__}")

# Make the repo's shared helpers importable
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from common.credentials import check_cds_credentials
check_cds_credentials()
print("CDS credentials found.")

In [ ]:
# Submit a minimal request to the CDS. The first call may take a few
# minutes while the request is queued and processed server-side.
output_path = Path(OUTPUT_DIR) / OUTPUT_FILENAME
output_path.parent.mkdir(parents=True, exist_ok=True)

request = {
    "product_type": ["reanalysis"],
    "variable": VARIABLES,
    "year": YEARS,
    "month": MONTHS,
    "day": DAYS,
    "time": HOURS,
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": BBOX,
}

client = cdsapi.Client()
client.retrieve("reanalysis-era5-single-levels", request).download(str(output_path))
print(f"Saved: {output_path} ({output_path.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# Open with xarray and print a dataset summary.
# Variables appear under GRIB short names (e.g. t2m for 2m_temperature)
# when netcdf is produced from the underlying GRIB archive.
ds = xr.open_dataset(output_path)
print(ds)

In [ ]:
# Plot 2m temperature as a map. Convert from kelvin to celsius for
# readability.
t2m_var = "t2m" if "t2m" in ds.data_vars else list(ds.data_vars)[0]
t2m = ds[t2m_var].squeeze()
t2m_celsius = t2m - 273.15

fig = plt.figure(figsize=(10, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

mesh = t2m_celsius.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdBu_r",
    cbar_kwargs={"label": "2 m temperature (C)"},
)

ax.coastlines(resolution="50m", linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor="gray")
gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
gl.top_labels = False
gl.right_labels = False

ax.set_title(f"ERA5 2 m temperature, {YEARS[0]}-{MONTHS[0]}-{DAYS[0]} {HOURS[0]} UTC")
plt.tight_layout()
plt.show()

## Next steps

- Read `docs/era5-single-levels/README.md` for the full dataset reference,
  licence details, citation, and known issues.
- See `docs/era5-single-levels/variables.md` for the full variable list.
- Expand the request: edit `VARIABLES`, `YEARS`, `MONTHS`, `DAYS`, `HOURS`,
  and `BBOX` in the config cell. Remember that request size is limited and
  large requests may take hours.
- For pre-computed daily statistics, see the ERA5 daily statistics entry
  in this repo instead of aggregating hourly data yourself.
- For upper-air variables on pressure levels, see the ERA5 pressure levels
  entry.